# Silver Layer - Earthquake Data

This notebook reads raw GeoJSON earthquake data from the bronze layer (`etl_restapi.bronze.earthquake_data` volume),
flattens the nested JSON structure into a tabular format, and writes the result to a Delta table in the silver schema.

**Source**: USGS Earthquake Hazards Program - GeoJSON Summary Feed
**Bronze**: JSON files in `/Volumes/etl_restapi/bronze/earthquake_data/`
**Silver**: Delta table `etl_restapi.silver.earthquake_data`

In [0]:
catalog_name = "etl_restapi"
bronze_schema = "bronze"
silver_schema = "silver"
volume_name = "earthquake_data"
silver_table = "earthquake_data"

bronze_path = f"/Volumes/{catalog_name}/{bronze_schema}/{volume_name}/"
print(f"Reading from: {bronze_path}")

dbutils.widgets.text("catalog_name", catalog_name, "etl_restapi")
catalog_name = dbutils.widgets.get("catalog_name")

Reading from: /Volumes/etl_restapi/bronze/earthquake_data/


In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType,
    IntegerType, BooleanType, ArrayType, MapType
)

# Define the schema for the USGS GeoJSON feed
feature_schema = StructType([
    StructField("type", StringType(), True),
    StructField("id", StringType(), True),
    StructField("properties", StructType([
        StructField("mag", DoubleType(), True),
        StructField("place", StringType(), True),
        StructField("time", LongType(), True),
        StructField("updated", LongType(), True),
        StructField("tz", IntegerType(), True),
        StructField("url", StringType(), True),
        StructField("detail", StringType(), True),
        StructField("felt", IntegerType(), True),
        StructField("cdi", DoubleType(), True),
        StructField("mmi", DoubleType(), True),
        StructField("alert", StringType(), True),
        StructField("status", StringType(), True),
        StructField("tsunami", IntegerType(), True),
        StructField("sig", IntegerType(), True),
        StructField("net", StringType(), True),
        StructField("code", StringType(), True),
        StructField("ids", StringType(), True),
        StructField("sources", StringType(), True),
        StructField("types", StringType(), True),
        StructField("nst", IntegerType(), True),
        StructField("dmin", DoubleType(), True),
        StructField("rms", DoubleType(), True),
        StructField("gap", DoubleType(), True),
        StructField("magType", StringType(), True),
        StructField("type", StringType(), True),
        StructField("title", StringType(), True),
    ]), True),
    StructField("geometry", StructType([
        StructField("type", StringType(), True),
        StructField("coordinates", ArrayType(DoubleType()), True),
    ]), True),
])

geojson_schema = StructType([
    StructField("type", StringType(), True),
    StructField("metadata", MapType(StringType(), StringType()), True),
    StructField("features", ArrayType(feature_schema), True),
    StructField("bbox", ArrayType(DoubleType()), True),
])

print("Schema defined successfully.")

Schema defined successfully.


In [0]:
from pyspark.sql.functions import (
    col, explode, from_json, to_timestamp, element_at, size,
    lit, current_timestamp, input_file_name, regexp_extract
)

# Read all JSON files from the bronze volume as raw text
raw_df = spark.read.text(bronze_path)

# Parse the GeoJSON using the defined schema
parsed_df = raw_df.select(
    from_json(col("value"), geojson_schema).alias("geojson")
)

# Explode the features array into individual rows
features_df = parsed_df.select(
    explode(col("geojson.features")).alias("feature")
)

print(f"Total features across all files: {features_df.count()}")

Total features across all files: 417


In [0]:
from pyspark.sql.functions import (
    col, to_timestamp, element_at, lit, current_timestamp, when
)

# Flatten properties and geometry coordinates into a tabular structure
silver_df = features_df.select(
    # Event identifiers
    col("feature.id").alias("event_id"),
    col("feature.properties.code").alias("event_code"),
    col("feature.properties.net").alias("network"),
    col("feature.properties.type").alias("event_type"),
    col("feature.properties.title").alias("title"),
    col("feature.properties.place").alias("place"),

    # Magnitude details
    col("feature.properties.mag").alias("magnitude"),
    col("feature.properties.magType").alias("magnitude_type"),

    # Timestamps (USGS provides milliseconds since epoch)
    to_timestamp((col("feature.properties.time") / 1000).cast("long")).alias("event_time"),
    to_timestamp((col("feature.properties.updated") / 1000).cast("long")).alias("updated_time"),

    # Coordinates: [longitude, latitude, depth]
    col("feature.geometry.type").alias("geometry_type"),
    element_at(col("feature.geometry.coordinates"), 1).alias("longitude"),
    element_at(col("feature.geometry.coordinates"), 2).alias("latitude"),
    element_at(col("feature.geometry.coordinates"), 3).alias("depth_km"),

    # Impact / felt reports
    col("feature.properties.felt").alias("felt_reports"),
    col("feature.properties.cdi").alias("cdi"),
    col("feature.properties.mmi").alias("mmi"),
    col("feature.properties.alert").alias("alert_level"),
    col("feature.properties.sig").alias("significance"),
    col("feature.properties.tsunami").alias("tsunami_flag"),

    # Seismic measurement details
    col("feature.properties.nst").alias("nst"),
    col("feature.properties.dmin").alias("dmin"),
    col("feature.properties.rms").alias("rms"),
    col("feature.properties.gap").alias("gap"),

    # Status and URLs
    col("feature.properties.status").alias("status"),
    col("feature.properties.url").alias("event_url"),
    col("feature.properties.detail").alias("detail_url"),
    col("feature.properties.ids").alias("ids"),
    col("feature.properties.sources").alias("sources"),
    col("feature.properties.types").alias("types"),

    # Metadata
    current_timestamp().alias("ingestion_time"),
)

print(f"Silver DataFrame row count: {silver_df.count()}")
silver_df.printSchema()
display(silver_df.limit(10))

Silver DataFrame row count: 417
root
 |-- event_id: string (nullable = true)
 |-- event_code: string (nullable = true)
 |-- network: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- place: string (nullable = true)
 |-- magnitude: double (nullable = true)
 |-- magnitude_type: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- updated_time: timestamp (nullable = true)
 |-- geometry_type: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- depth_km: double (nullable = true)
 |-- felt_reports: integer (nullable = true)
 |-- cdi: double (nullable = true)
 |-- mmi: double (nullable = true)
 |-- alert_level: string (nullable = true)
 |-- significance: integer (nullable = true)
 |-- tsunami_flag: integer (nullable = true)
 |-- nst: integer (nullable = true)
 |-- dmin: double (nullable = true)
 |-- rms: double (nullable = true)
 |-- gap: double (nullable

event_id,event_code,network,event_type,title,place,magnitude,magnitude_type,event_time,updated_time,geometry_type,longitude,latitude,depth_km,felt_reports,cdi,mmi,alert_level,significance,tsunami_flag,nst,dmin,rms,gap,status,event_url,detail_url,ids,sources,types,ingestion_time
tx2026qwdcwr,2026qwdcwr,tx,earthquake,"M 1.3 - 16 km NW of Midland, Texas","16 km NW of Midland, Texas",1.3,ml,2026-08-27T16:14:05.000Z,2026-08-27T16:18:20.000Z,Point,-102.215,32.089,4.1298,null,null,null,null,26,0,43,0.0,0.4,56.0,automatic,https://earthquake.usgs.gov/earthquakes/eventpage/tx2026qwdcwr,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/tx2026qwdcwr.geojson,",tx2026qwdcwr,",",tx,",",origin,phase-data,",2026-09-01T15:52:31.461Z
ci41537544,41537544,ci,earthquake,"M 1.3 - 4 km NNE of Beaumont, CA","4 km NNE of Beaumont, CA",1.29,ml,2026-08-27T15:57:47.000Z,2026-08-27T16:08:30.000Z,Point,-116.9615,33.9631666666667,8.3,null,null,null,null,26,0,61,0.1318,0.15,23.0,automatic,https://earthquake.usgs.gov/earthquakes/eventpage/ci41537544,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/ci41537544.geojson,",ci41537544,",",ci,",",focal-mechanism,nearby-cities,origin,phase-data,",2026-09-01T15:52:31.461Z
tx2026qwcnkr,2026qwcnkr,tx,earthquake,"M 1.5 - 10 km N of Midland, Texas","10 km N of Midland, Texas",1.5,ml,2026-08-27T15:56:09.000Z,2026-08-27T15:59:37.000Z,Point,-102.074,32.091,8.0258,null,null,null,null,35,0,14,0.0,0.5,115.0,automatic,https://earthquake.usgs.gov/earthquakes/eventpage/tx2026qwcnkr,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/tx2026qwcnkr.geojson,",tx2026qwcnkr,",",tx,",",origin,phase-data,",2026-09-01T15:52:31.461Z
ci41537536,41537536,ci,earthquake,"M 1.0 - 12 km WSW of Salton City, CA","12 km WSW of Salton City, CA",0.98,ml,2026-08-27T15:48:01.000Z,2026-08-27T15:51:28.000Z,Point,-116.0665,33.2428333333333,4.34,null,null,null,null,15,0,38,0.07712,0.23,55.0,automatic,https://earthquake.usgs.gov/earthquakes/eventpage/ci41537536,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/ci41537536.geojson,",ci41537536,",",ci,",",nearby-cities,origin,phase-data,",2026-09-01T15:52:31.461Z
ok2026quax,2026quax,ok,earthquake,"M 1.3 - 3 km ENE of Medford, Oklahoma","3 km ENE of Medford, Oklahoma",1.3,ml,2026-08-27T15:45:19.000Z,2026-08-27T16:08:52.000Z,Point,-97.70416667,36.82016667,7.16,null,null,null,null,26,0,36,0.1286747496,0.16,79.0,reviewed,https://earthquake.usgs.gov/earthquakes/eventpage/ok2026quax,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/ok2026quax.geojson,",ok2026quax,",",ok,",",origin,phase-data,",2026-09-01T15:52:31.461Z
uu80150771,80150771,uu,earthquake,"M 0.8 - 17 km NE of Milford, Utah","17 km NE of Milford, Utah",0.79,ml,2026-08-27T15:44:46.000Z,2026-08-27T16:07:21.000Z,Point,-112.893,38.519,2.06,null,null,null,null,10,0,13,0.009597,0.08,116.0,reviewed,https://earthquake.usgs.gov/earthquakes/eventpage/uu80150771,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/uu80150771.geojson,",uu80150771,",",uu,",",origin,phase-data,",2026-09-01T15:52:31.461Z
aka2026qyunul,a2026qyunul,ak,earthquake,"M 0.8 - 93 km WNW of Karluk, Alaska","93 km WNW of Karluk, Alaska",0.8,ml,2026-08-27T15:39:24.000Z,2026-08-27T15:40:06.000Z,Point,-155.879,57.926,16.2,null,null,null,null,10,0,11,0.2,0.3,170.0,automatic,https://earthquake.usgs.gov/earthquakes/eventpage/aka2026qyunul,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/aka2026qyunul.geojson,",aka2026qyunul,",",ak,",",origin,phase-data,",2026-09-01T15:52:31.461Z
us7000tca4,7000tca4,us,earthquake,"M 3.5 - 10 km NW of Mérida, Venezuela","10 km NW of Mérida, Venezuela",3.5,ml,2026-08-27T15:36:05.000Z,2026-08-27T16:42:23.000Z,Point,-71.214,8.6667,10.0,2,3.9,null,null,189,0,7,0.612,0.8,144.0,reviewed,https://earthquake.usgs.gov/earthquakes/eventpage/us7000tca4,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us7000tca4.geojson,",us7000tca4,",",us,",",dyfi,origin,phase-data,",2026-09-01T15:52:31.461Z
nc75425982,75425982,nc,earthquake,"M 1.

In [0]:
# Ensure the silver schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}")

# Write the flattened data to the silver Delta table
silver_table_name = f"{catalog_name}.{silver_schema}.{silver_table}"

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table_name)
)

print(f"Silver table written to: {silver_table_name}")

Silver table written to: etl_restapi.silver.earthquake_data


In [0]:
# Validate the silver table - check actual row count
full_count = spark.sql(f"SELECT COUNT(*) as total FROM {silver_table_name}").collect()[0]['total']
print(f"Total rows in silver table: {full_count}")

if full_count == 0:
    raise Exception("Silver table is empty - check bronze layer data.")

print(f"\nFetching all {full_count} records from the silver table...")

# Display all records (sorted by magnitude)
display(spark.sql(f"""
    SELECT 
        event_id, title, place, magnitude, magnitude_type,
        event_time, longitude, latitude, depth_km,
        felt_reports, alert_level, tsunami_flag, status
    FROM {silver_table_name}
    ORDER BY magnitude DESC
"""))

Total rows in silver table: 417

Fetching all 417 records from the silver table...


event_id,title,place,magnitude,magnitude_type,event_time,longitude,latitude,depth_km,felt_reports,alert_level,tsunami_flag,status
us7000tdav,"M 5.6 - 82 km SE of Maba, Indonesia","82 km SE of Maba, Indonesia",5.6,mb,2026-09-01T08:22:20.000Z,128.8054,0.1685,10.0,null,green,0,reviewed
us7000tc18,"M 5.6 - 209 km ESE of Ōfunato, Japan","209 km ESE of Ōfunato, Japan",5.6,mww,2026-08-26T19:21:42.000Z,143.8447,38.1821,16.05,1,green,0,reviewed
us7000tc8j,"M 5.3 - 54 km NNE of Labuan Bajo, Indonesia","54 km NNE of Labuan Bajo, Indonesia",5.3,mb,2026-08-27T10:33:24.000Z,120.0539,-8.0324,10.0,null,null,0,reviewed
us7000tc7l,"M 5.3 - Volcano Islands, Japan region","Volcano Islands, Japan region",5.3,mb,2026-08-27T05:44:26.000Z,142.7624,24.2906,10.0,null,null,0,reviewed
us7000tdab,"M 5.3 - 89 km SSW of Nikolski, Alaska","89 km SSW of Nikolski, Alaska",5.3,mww,2026-09-01T06:44:36.000Z,-169.4456,52.2132,26.001,3,null,0,reviewed
us7000tdbr,"M 5.2 - 35 km SE of Sarangani, Philippines","35 km SE of Sarangani, Philippines",5.2,mww,2026-09-01T13:22:30.000Z,125.672,5.1623,120.779,9,null,0,reviewed
us7000tc56,M 5.1 - South Atlantic Ocean,South Atlantic Ocean,5.1,mb,2026-08-27T04:19:08.000Z,-20.2977,-22.5672,10.0,null,null,0,reviewed
us7000tc9m,M 5.0 - western Xizang,western Xizang,5.0,mb,2026-08-27T14:45:48.000Z,86.9395,33.2423,10.0,null,null,0,reviewed
us7000tdbc,"M 4.9 - 117 km WNW of Pangai, Tonga","117 km WNW of Pangai, Tonga",4.9,mb,2026-09-01T10:34:48.000Z,-175.4039,-19.4439,198.068,null,null,0,reviewed
us7000tc0k,"M 4.9 - 42 km NE of Ruteng, Indonesia","42 km NE of Ruteng, Indonesia",4.9,mb,2026-08-26T18:23:13.000Z,120.6979,-8.3107,10.0,null,null,0,reviewed
